# 🧠 NEURO-CUT // Phase 3: PPO Policy 5,000-Episode Overnight Training Run

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AmanM006/neurocut/blob/main/notebooks/train_ppo_colab.ipynb)

This notebook runs the **Resumable 5,000-Episode Overnight PPO Training Run** for **Neuro-Cut** (Google Cloud Agentic Cinema Hackathon).

### Key Features:
1. **Auto Checkpoint-Resume**: On startup, it checks ClickHouse Cloud for the highest existing `ppo_overnight_big_ep_{N}`. If found, it automatically loads `ppo_best.npz` and resumes from episode `N + 1`! Zero lost progress on disconnects.
2. **5,000 Episode Exploration**: ~4.8 hours on NVIDIA T4 GPU (at ~3.5s/ep), providing deep value convergence and multi-step policy refinement.
3. **Real ClickHouse Cloud Ingestion**: Streams all attempts and rewards into `default.edit_attempts` at `fwybcmwtlx.asia-southeast1.gcp.clickhouse.cloud`.
4. **Live Dashboard Observability**: Your local Next.js dashboard (**Panel D: PPO RL Policy Training**) dynamically plots the 20-episode rolling average curve in real time!

### Step 1: Install Dependencies & Clone Repository

In [ ]:
!apt-get update -qq && apt-get install -y -qq ffmpeg
!pip install -q clickhouse-connect pydantic pydantic-settings opencv-python pillow torch

import os
import sys
import subprocess

if not os.path.exists('/content/neurocut'):
    subprocess.run(['git', 'clone', 'https://github.com/AmanM006/neurocut.git', '/content/neurocut'], check=True)
else:
    subprocess.run(['git', '-C', '/content/neurocut', 'pull', 'origin', 'main'], check=False)

os.environ['CLICKHOUSE_HOST'] = 'fwybcmwtlx.asia-southeast1.gcp.clickhouse.cloud'
os.environ['CLICKHOUSE_PORT'] = '8443'
os.environ['CLICKHOUSE_USER'] = 'default'
os.environ['CLICKHOUSE_PASSWORD'] = ''
os.environ['CLICKHOUSE_DATABASE'] = 'default'
os.environ['CLICKHOUSE_SECURE'] = 'true'

if '/content/neurocut' not in sys.path:
    sys.path.insert(0, '/content/neurocut')

# Invalidate cached imports if session was active
for mod in list(sys.modules.keys()):
    if mod.startswith('backend') or mod.startswith('scripts'):
        del sys.modules[mod]

print('>>> Environment ready! Connected to ClickHouse Cloud at fwybcmwtlx.asia-southeast1.gcp.clickhouse.cloud')

### Step 2: Verify ClickHouse Cloud & Query Baseline

In [ ]:
import clickhouse_connect

print('>>> Connecting to ClickHouse Cloud...')
ch_client = clickhouse_connect.get_client(
    host='fwybcmwtlx.asia-southeast1.gcp.clickhouse.cloud',
    port=8443,
    user='default',
    password='',
    secure=True
)
rows = ch_client.query('SELECT count() FROM default.edit_attempts').result_set[0][0]
print(f'Connected! Existing edit attempts in ClickHouse: {rows}')

baseline_row = ch_client.query("""
    SELECT episode_id, max(reward) as r 
    FROM default.edit_attempts 
    WHERE episode_id = 'beam_search_baseline' 
    GROUP BY episode_id
""").result_set

if baseline_row:
    print(f'🎯 Target Baseline to Beat: {baseline_row[0][0]} -> Reward: {baseline_row[0][1]:.4f}')
else:
    print('🎯 Target Baseline to Beat: 0.6730 (Beam Search Baseline)')

### Step 3: Run Resumable 5,000-Episode Overnight PPO Training Run
Auto-checks ClickHouse for existing progress and resumes automatically if interrupted.

In [ ]:
import os
import sys
import time
import numpy as np
from pathlib import Path

os.environ['CLICKHOUSE_HOST'] = 'fwybcmwtlx.asia-southeast1.gcp.clickhouse.cloud'
os.environ['CLICKHOUSE_PORT'] = '8443'
os.environ['CLICKHOUSE_USER'] = 'default'
os.environ['CLICKHOUSE_PASSWORD'] = ''
os.environ['CLICKHOUSE_DATABASE'] = 'default'
os.environ['CLICKHOUSE_SECURE'] = 'true'

if '/content/neurocut' not in sys.path:
    sys.path.insert(0, '/content/neurocut')

from backend.optimizer.ppo_agent import PPOAgent
from backend.clickhouse.client import clickhouse_client

# Overnight Configuration
TOTAL_EPISODES = 5000
STEPS_PER_EPISODE = 4
CHECKPOINT_INTERVAL = 50
EPISODE_PREFIX = 'ppo_overnight_big_ep_'

MODELS_DIR = Path('/content/neurocut/backend/models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)
best_checkpoint_path = str(MODELS_DIR / 'ppo_best.npz')

# Auto-Resume Check: Find highest logged episode in ClickHouse Cloud
print('>>> Checking ClickHouse Cloud for previous overnight progress...')
resume_q = f"SELECT DISTINCT episode_id FROM default.edit_attempts WHERE episode_id LIKE '{EPISODE_PREFIX}%'"
resume_res = clickhouse_client.query(resume_q)
eps = []
for r in resume_res:
    try:
        eps.append(int(r['episode_id'].split('_')[-1]))
    except Exception:
        pass
last_ep = max(eps) if eps else 0

start_ep = last_ep + 1
agent = PPOAgent(episode_id=f'{EPISODE_PREFIX}{start_ep}')
agent.scorer.gemini_client = None  # Fast deterministic scoring for RL rollouts

if last_ep > 0:
    print(f'*** AUTO-RESUMING FROM EPISODE {start_ep}/{TOTAL_EPISODES} (Previous logged: {last_ep}) ***')
    if os.path.exists(best_checkpoint_path):
        agent.load_checkpoint(best_checkpoint_path)
        print(f'*** Loaded best checkpoint from {best_checkpoint_path} ***')
else:
    print(f'*** Starting fresh overnight training run from Episode 1 to {TOTAL_EPISODES} ***')

print('=' * 70)
print('      NEURO-CUT // 5,000-EPISODE OVERNIGHT PPO WORKER')
print(f'      Range: Ep {start_ep} -> {TOTAL_EPISODES} | Steps/Ep: {STEPS_PER_EPISODE} | Checkpoint: Every {CHECKPOINT_INTERVAL} eps')
print('=' * 70)

best_eval_reward = -999.0
history = []
start_time = time.time()

for ep in range(start_ep, TOTAL_EPISODES + 1):
    ep_id = f'{EPISODE_PREFIX}{ep}'
    initial_reward = agent.reset_episode(ep_id)
    ep_rewards = [initial_reward]

    for step in range(STEPS_PER_EPISODE):
        step_res = agent.optimize_step(compile_video=False, deterministic=False)
        ep_rewards.append(step_res['reward'])

    train_metrics = agent.train_step()
    final_ep_reward = max(ep_rewards)
    history.append(final_ep_reward)

    rolling_avg = float(np.mean(history[-20:]))

    if ep % 10 == 0 or ep == start_ep or ep == TOTAL_EPISODES:
        elapsed = time.time() - start_time
        avg_pace = elapsed / max(1, (ep - start_ep + 1))
        remaining_sec = avg_pace * (TOTAL_EPISODES - ep)
        print(f'Ep {ep:>4}/{TOTAL_EPISODES} | Final: {final_ep_reward:.4f} | '
              f'Rolling(20): {rolling_avg:.4f} | Loss: {train_metrics.get("loss", 0):.4f} | '
              f'Elapsed: {elapsed/60:.1f}m | ETA: {remaining_sec/60:.1f}m')

    if ep % CHECKPOINT_INTERVAL == 0 or ep == TOTAL_EPISODES:
        ckpt_path = str(MODELS_DIR / f'ppo_checkpoint_ep{ep}.npz')
        agent.save_checkpoint(ckpt_path)

        # Deterministic evaluation on fresh held-out episode
        eval_ep_id = f'ppo_eval_overnight_ep{ep}'
        agent.reset_episode(eval_ep_id)
        eval_rewards = []
        for _ in range(STEPS_PER_EPISODE):
            ev = agent.optimize_step(compile_video=False, deterministic=True)
            eval_rewards.append(ev['reward'])
        eval_final = max(eval_rewards) if eval_rewards else 0.0
        print(f'  >>> Checkpoint Ep {ep} Deterministic Eval: {eval_final:.4f} (Previous Best: {best_eval_reward:.4f})')

        if eval_final > best_eval_reward:
            best_eval_reward = eval_final
            agent.save_checkpoint(best_checkpoint_path)
            print(f'  *** NEW BEST MODEL SAVED! Eval: {best_eval_reward:.4f} ***')

        agent.buffer.clear()

total_time = time.time() - start_time
print('\n' + '=' * 70)
print(f'      OVERNIGHT PPO TRAINING COMPLETED IN {total_time/3600:.2f} HOURS ({total_time/60:.1f} min)')
print(f'      Best Deterministic Eval Reward: {best_eval_reward:.4f}')
print('=' * 70)

### Step 4: Run Final Frozen-Policy Benchmark on `ppo_final_eval`
Evaluates the best checkpoint deterministically and logs to `ppo_final_eval` in ClickHouse Cloud.

In [ ]:
import os
import sys
from pathlib import Path

if '/content/neurocut' not in sys.path:
    sys.path.insert(0, '/content/neurocut')

from backend.optimizer.ppo_agent import PPOAgent
from backend.clickhouse.client import clickhouse_client

print('\n[Step 4] Executing Final Frozen Policy Evaluation on \'ppo_final_eval\'...')
eval_agent = PPOAgent(episode_id='ppo_final_eval')
eval_agent.scorer.gemini_client = None
loaded = eval_agent.load_checkpoint(best_checkpoint_path)
print(f'  * Best checkpoint loaded ({best_checkpoint_path}): {loaded}')

eval_init = eval_agent.reset_episode('ppo_final_eval')
final_rewards = [eval_init]

for s in range(STEPS_PER_EPISODE):
    ev_step = eval_agent.optimize_step(compile_video=(s == STEPS_PER_EPISODE - 1), deterministic=True)
    final_rewards.append(ev_step['reward'])
    print(f'  Eval Step #{s+1} | Action: {ev_step["action"]:^14} on {str(ev_step["target_clip_id"]):^24} | '
          f'Reward: {ev_step["reward"]:.4f} | Verdict: {ev_step["verdict"]}')

ppo_final_reward = max(final_rewards)
print(f'\n>>> PPO Frozen Policy Final Evaluated Reward: {ppo_final_reward:.4f}')

print('\n' + '=' * 70)
print('      CLICKHOUSE CLOUD HEAD-TO-HEAD BENCHMARK')
print('=' * 70)
query = '''
SELECT episode_id, max(reward) as final_reward 
FROM default.edit_attempts 
WHERE episode_id IN ('beam_search_baseline', 'ppo_final_eval')
GROUP BY episode_id
ORDER BY final_reward DESC
'''
rows = clickhouse_client.query(query)
for r in rows:
    tag = '  <-- CURRENT CHAMPION' if r['final_reward'] == max(x['final_reward'] for x in rows) else ''
    print(f'  * Episode: {r["episode_id"]:^24} | Final Reward: {r["final_reward"]:.4f}{tag}')

### Step 5: Visualize the 20-Episode Rolling Learning Curve

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5), dpi=120)
episodes = np.arange(1, len(history) + 1)
rolling = [np.mean(history[max(0, i-20):i+1]) for i in range(len(history))]

plt.plot(episodes, history, alpha=0.20, color='cyan', label='Episode Raw Reward')
plt.plot(episodes, rolling, color='#00d4ff', linewidth=2.5, label='20-Episode Rolling Average')
plt.axhline(0.6730, color='#ffaa00', linestyle='--', linewidth=2, label='Beam Search Baseline (0.6730)')
plt.axhline(0.5000, color='#888888', linestyle=':', linewidth=1.5, label='Rough Cut Initial (0.5000)')

plt.title('NEURO-CUT: 5,000-Episode Overnight PPO Learning Progression', fontsize=13, fontweight='bold')
plt.xlabel('Episode', fontsize=11)
plt.ylabel('Scalar Retention Reward', fontsize=11)
plt.legend(loc='lower right')
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

### Step 6: Download Best Trained Weights (`ppo_best.npz`)

In [ ]:
from google.colab import files
if os.path.exists(best_checkpoint_path):
    print(f'>>> Downloading {best_checkpoint_path}...')
    files.download(best_checkpoint_path)
else:
    print('Checkpoint not found!')